In [ ]:
import sys
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import sys
import numpy as np
from PIL import Image

from google.colab import drive
drive.mount('/content/drive')

sys.path.insert(0, '/content/drive/MyDrive/video_surgery')

from src import config, setup
setup.prep_env()
setup.download_models()
sys.path.insert(0, config.COLAB_GROUNDED_SAM_DIR)
sys.path.insert(0, config.COLAB_RAFT_DIR / 'core')

raise SystemExit('\nRestart Session to continue')


In [ ]:
from src import helpers

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
gd_processor, gd_model = helpers.load_groundingdino(device)
sam2_predictor = helpers.load_sam2_image(device)

In [ ]:
image_pil = Image.open(config.ANCHOR_PATH).convert("RGB")

text_labels = [[config.specs['prompt']]]

inputs = gd_processor(
    images=image_pil,
    text=text_labels,
    return_tensors='pt'
).to(device)

with torch.inference_mode():
    outputs = gd_model(**inputs)

results = gd_processor.post_process_grounded_object_detection(
    outputs,
    inputs.input_ids,
    threshold=0.3,
    text_threshold=0.25,
    target_sizes=[image_pil.size[::-1]]
)[0]

boxes  = results['boxes'].cpu().numpy()
scores = results['scores'].cpu().numpy()
labels = results['text_labels']

if len(boxes) == 0:
    raise ValueError(
        'No objects are found'
    )

print(f"Detections: {len(boxes)}")
for i, (box, score, label) in enumerate(zip(boxes, scores, labels)):
    print(f"  [{i}] {label} - score: {score:.3f} - box: {box.astype(int)}")

In [ ]:
best_idx = scores.argmax()
best_box = boxes[best_idx]
print(f"Using box:   {best_box.astype(int)} - score: {scores[best_idx]:.3f}")

image_np = np.array(image_pil)

with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
    sam2_predictor.set_image(image_np)
    masks, scores_sam, logits = sam2_predictor.predict(
        box=best_box,
        multimask_output=True
    )

best_mask_idx = scores_sam.argmax()
mask = masks[best_mask_idx].astype(np.uint8)

print(f"Mask shape:  {mask.shape}")
print(f"Masked pixels: {mask.sum()} / {mask.size}, {100 * mask.sum() / mask.size:.2f}%")

In [ ]:
ANCHOR_MASK_PATH = config.MASKS / '0.npy'
np.save(ANCHOR_MASK_PATH, mask)
print(f"Anchor mask saved to {ANCHOR_MASK_PATH}")

In [ ]:
image_bgr = cv2.imread(config.ANCHOR_PATH)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

overlay = image_rgb.copy().astype(np.float32)
overlay[mask == 1] = overlay[mask == 1] * 0.5 + np.array([255, 0, 0]) * 0.5
overlay = overlay.astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image_rgb)
x1, y1, x2, y2 = best_box.astype(int)
rect = patches.Rectangle(
    (x1, y1), x2 - x1, y2 - y1,
    linewidth=2, edgecolor='lime', facecolor='none'
)
axes[0].add_patch(rect)
axes[0].set_title(f"GroundingDINO box\n{config.specs['prompt']} "
                  f"(score: {scores[best_idx]:.2f})")
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title('SAM 2 binary mask')
axes[1].axis('off')

axes[2].imshow(overlay)
axes[2].set_title('Mask overlay on frame 0')
axes[2].axis('off')

plt.tight_layout()
plt.show()